# 초기 커리큘럼 생성을 위한 GraphDB 구축
- 구축 순서
    - Wikipedia 데이터
    - 기존 데이터셋
    - 논문 추출 데이터셋
- 특징
    - 논문 추출 데이터셋의 경우, Keyword 자체가 기존에 구축된 Node와 의미적으로 중복되는 경우가 있다. (LLM 기반 추출을 사용) 따라서, 논문 추출 데이터셋의 Keyword를 넣을 때, 기존 노드의 Alias와 중복되는지를 확인해, 중복되면 기존 노드에 병합하고 중복되지 않으면 신규 노드를 생성한다.
    - Paper-Keyword간 Edge 또한 Alias를 기반으로 자동으로 연결되도록 한다.

## 1. Import Library

In [ ]:
import pandas as pd
from neo4j import GraphDatabase
import ast
from tqdm.auto import tqdm
from typing import List, Dict, Optional, Tuple

## 2. Connect Neo4j

In [ ]:
NEO4J_URI=""
NEO4J_USERNAME=""
NEO4J_PASSWORD=""

In [ ]:
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))

def close_driver():
    driver.close()

print(driver.get_server_info())

## 3. Create Utility Functions

In [ ]:
def parse_to_list(value):
    if isinstance(value, list):
        return value
    if pd.isna(value) or value == '':
        return []
    if not isinstance(value, str):
        return []

    value = value.strip()
    
    if value.startswith('[') and value.endswith(']'):
        try:
            return ast.literal_eval(value)
        except (ValueError, SyntaxError):
            pass
    
    # ", "로 구분된 문자열 처리
    return [item.strip() for item in value.split(',')]

## 4. Database Constraint, Index 생성

In [ ]:
def create_constraints(tx):
    # CONSTRAINT
    try:
        tx.run("CREATE CONSTRAINT keyword_name_unique IF NOT EXISTS FOR (k:Keyword) REQUIRE k.name IS UNIQUE")
        tx.run("CREATE CONSTRAINT paper_name_unique IF NOT EXISTS FOR (p:Paper) REQUIRE p.name IS UNIQUE")
        print("Constraint 생성 완료")
    except Exception as e:
        print(f"{e}")

def create_indexes(tx):
    # INDEX
    try:
        tx.run("CREATE INDEX keyword_name_index IF NOT EXISTS FOR (k:Keyword) ON (k.name)")
        tx.run("CREATE INDEX paper_name_index IF NOT EXISTS FOR (p:Paper) ON (p.name)")
        tx.run("CREATE INDEX paper_paperId_index IF NOT EXISTS FOR (p:Paper) ON (p.paperId)")
        print("Index 생성 완료")
    except Exception as e:
        print(f"{e}")

with driver.session() as session:
    session.execute_write(create_constraints)
    session.execute_write(create_indexes)

## 5. 기존 데이터셋 구축

### 5-1. Query 함수

In [ ]:
def create_keyword_nodes_batch(tx, nodes_batch):
    query = """
    UNWIND $nodes as node
    MERGE (k:Keyword {name: node.name})
    SET k.link = node.link,
        k.alias = node.alias,
        k.categories = node.categories
    """
    tx.run(query, nodes=nodes_batch)

def create_prereq_relation_batch(tx, edges_batch):
    query = """
    UNWIND $edges as edge
    MATCH (source:Keyword {name: edge.source_name})
    MATCH (target:Keyword {name: edge.target_name})
    MERGE (source)-[r:PREREQ]->(target)
    SET r.strength = edge.strength,
        r.reason = edge.reason
    """
    tx.run(query, edges=edges_batch)

In [ ]:
def create_nodes(nodes_df, driver):
    batch_size = 1000
    total_nodes = len(nodes_df)

    with driver.session() as session:
        for i in range(0, total_nodes, batch_size):
            batch = nodes_df.iloc[i:i+batch_size]
            nodes_batch = [
                {
                    'name': row['name'],
                    'link': row['link'],
                    'alias': row['alias'],
                    'categories': row['categories']
                }
                for _, row in batch.iterrows()
            ]
            session.execute_write(create_keyword_nodes_batch, nodes_batch)
            print(f"노드 생성: {min(i+batch_size, total_nodes)}/{total_nodes}")

    print(f"\n총 생성 노드: {total_nodes}")


def create_edges(edges_df, driver):
    batch_size = 1000
    total_edges = len(edges_df)

    with driver.session() as session:
        for i in range(0, total_edges, batch_size):
            batch = edges_df.iloc[i:i+batch_size]
            edges_batch = [
                {
                    'source_name': row['prereq_name'],
                    'target_name': row['name'],
                    'strength': float(row['strength']),
                    'reason': row['reason'] if pd.notna(row['reason']) else None
                }
                for _, row in batch.iterrows()
            ]
            session.execute_write(create_prereq_relation_batch, edges_batch)
            print(f"엣지 생성: {min(i+batch_size, total_edges)}/{total_edges}")

    print(f"\n총 생성 엣지: {total_edges}")

### 5-2. Wikipedia 데이터셋

In [ ]:
NODE_DATA = '../data/wikipedia/wikipedia_node.csv'
EDGE_DATA = '../data/wikipedia/wikipedia_edge.csv'

nodes_df = pd.read_csv(NODE_DATA)
edges_df = pd.read_csv(EDGE_DATA)

print(f"노드 수: {len(nodes_df)}")
print(f"엣지 수: {len(edges_df)}")

print("\n=== 노드 샘플 ===")
print(nodes_df.head())

print("\n=== 엣지 샘플 ===")
print(edges_df.head())

In [ ]:
nodes_df['categories'] = nodes_df['categories'].apply(parse_to_list)
nodes_df['alias'] = nodes_df['alias'].apply(parse_to_list)
print(nodes_df.head())

In [ ]:
print("="*50)
create_nodes(nodes_df, driver)
print("="*50)
create_edges(edges_df, driver)

### 5-3. Metacademy 데이터셋

In [ ]:
NODE_DATA = '../data/metacademy/metacademy_node_with_categories.csv'
EDGE_DATA = '../data/metacademy/metacademy_edge.csv'

nodes_df = pd.read_csv(NODE_DATA)
edges_df = pd.read_csv(EDGE_DATA)

print(f"노드 수: {len(nodes_df)}")
print(f"엣지 수: {len(edges_df)}")

print("\n=== 노드 샘플 ===")
print(nodes_df.head())

print("\n=== 엣지 샘플 ===")
print(edges_df.head())

In [ ]:
nodes_df['categories'] = nodes_df['categories'].apply(parse_to_list)
nodes_df['alias'] = nodes_df['alias'].apply(parse_to_list)
print(nodes_df.head())

In [ ]:
print("="*50)
create_nodes(nodes_df, driver)
print("="*50)
create_edges(edges_df, driver)

### 5-4. Tutorialbank 데이터셋

In [ ]:
NODE_DATA = '../data/tutorialbank/tutorialbank_node_with_categories.csv'
EDGE_DATA = '../data/tutorialbank/tutorialbank_edge.csv'

nodes_df = pd.read_csv(NODE_DATA)
edges_df = pd.read_csv(EDGE_DATA)

print(f"노드 수: {len(nodes_df)}")
print(f"엣지 수: {len(edges_df)}")

print("\n=== 노드 샘플 ===")
print(nodes_df.head())

print("\n=== 엣지 샘플 ===")
print(edges_df.head())

In [ ]:
nodes_df['categories'] = nodes_df['categories'].apply(parse_to_list)
nodes_df['alias'] = nodes_df['alias'].apply(parse_to_list)
print(nodes_df.head())

In [ ]:
print("="*50)
create_nodes(nodes_df, driver)
print("="*50)
create_edges(edges_df, driver)

## 6. SemanticScholar Paper 데이터셋 구축

### 6-1. 데이터셋 로드

In [ ]:
NODE_KEYWORD_DATA = '../data/paper/papers_node_kc.csv'
NODE_PAPER_DATA = '../data/paper/papers_node_paper.csv'
EDGE_KEYWORD_DATA = '../data/paper/papers_edge_paper_kc.csv'
EDGE_PAPER_DATA = '../data/paper/papers_edge_paper_paper.csv'

nodes_keyword_df = pd.read_csv(NODE_KEYWORD_DATA)
nodes_paper_df = pd.read_csv(NODE_PAPER_DATA)
edges_keyword_df = pd.read_csv(EDGE_KEYWORD_DATA)
edges_paper_df = pd.read_csv(EDGE_PAPER_DATA)

print(f"키워드 노드 수: {len(nodes_keyword_df)}")
print(f"논문 노드 수: {len(nodes_paper_df)}")
print(f"논문-키워드 엣지 수: {len(edges_keyword_df)}")
print(f"논문-논문 엣지 수: {len(edges_paper_df)}")

# 데이터 확인
print("\n=== 노드 샘플 ===")
print(nodes_keyword_df.head())
print(nodes_paper_df.head())

print("\n=== 엣지 샘플 ===")
print(edges_keyword_df.head())
print(edges_paper_df.head())

In [ ]:
nodes_keyword_df['categories'] = nodes_keyword_df['categories'].apply(parse_to_list)
nodes_keyword_df['alias'] = nodes_keyword_df['alias'].apply(parse_to_list)
nodes_paper_df['categories'] = nodes_paper_df['categories'].apply(parse_to_list)

print("데이터 전처리 완료")
print(nodes_keyword_df.head())
print(nodes_paper_df.head())

In [ ]:
edges_paper_df['intents'] = edges_paper_df['intents'].apply(parse_to_list)
edges_paper_df['contexts'] = edges_paper_df['contexts'].apply(parse_to_list)

edges_paper_df.head()

### 6-2. Keyword 추가/병합 함수 정의

In [ ]:
def find_matching_keyword(tx, new_name: str, new_aliases: List[str]) -> Optional[str]:
    """
    새 Keyword의 name 또는 alias가 기존 Keyword 노드의 name이나 alias와 중복되는지 확인
    중복되면 기존 노드의 name 반환, 없으면 None 반환
    """
    search_terms = [new_name.lower().strip()] + [alias.lower().strip() for alias in new_aliases if alias]
    
    query = """
    MATCH (k:Keyword)
    WHERE k.name IS NOT NULL
      AND (
        toLower(k.name) IN $search_terms
        OR any(a IN k.alias WHERE toLower(a) IN $search_terms)
      )
    RETURN k.name as existing_name
    LIMIT 1
    """
    
    result = tx.run(query, search_terms=search_terms)
    record = result.single()
    return record['existing_name'] if record else None


def merge_to_existing_keyword(tx, existing_name: str, new_name: str, new_aliases: List[str], 
                              new_link: str, new_categories: List[str]) -> Dict:
    """
    기존 Keyword 노드에 새로운 name과 alias를 병합
    """
    aliases_to_add = [new_name] + new_aliases
    
    query = """
    MATCH (k:Keyword {name: $existing_name})
    WITH k, COALESCE(k.alias, []) as current_aliases,
         COALESCE(k.categories, []) as current_categories

    // 중복되지 않은 alias만 추가
    WITH k, current_aliases, current_categories,
         [a IN $new_aliases WHERE NOT a IN current_aliases] as unique_new_aliases
    SET k.alias = current_aliases + unique_new_aliases

    // link가 비어있으면 새 link로 업데이트
    SET k.link = CASE WHEN k.link IS NULL OR k.link = '' THEN $new_link ELSE k.link END

    // categories 병합 (중복 제거)
    WITH k, current_categories,
         [c IN $new_categories WHERE NOT c IN current_categories] as unique_new_categories
    SET k.categories = current_categories + unique_new_categories
    RETURN k.name as name, size(k.alias) as alias_count, 'merged' as action
    """
    
    result = tx.run(query, 
                   existing_name=existing_name,
                   new_aliases=aliases_to_add,
                   new_link=new_link,
                   new_categories=new_categories)
    return result.single().data()


def create_new_keyword(tx, name: str, link: str, aliases: List[str], categories: List[str]) -> Dict:
    """
    새로운 Keyword 노드 생성
    """
    query = """
    CREATE (k:Keyword {
        name: $name,
        link: $link,
        alias: $aliases,
        categories: $categories
    })
    RETURN k.name as name, size(k.alias) as alias_count, 'created' as action
    """
    
    result = tx.run(query, name=name, link=link, aliases=aliases, categories=categories)
    return result.single().data()


def process_keyword(tx, name: str, link: str, aliases: List[str], categories: List[str]) -> Dict:
    """
    Keyword를 하나씩 처리: 기존 노드가 있으면 병합, 없으면 생성
    """
    # 기존 노드 찾기
    existing_name = find_matching_keyword(tx, name, aliases)
    
    # 병합 또는 생성
    if existing_name:
        return merge_to_existing_keyword(tx, existing_name, name, aliases, link, categories)
    else:
        return create_new_keyword(tx, name, link, aliases, categories)

### 6-3. Keyword Node 생성

In [ ]:
# 통계 변수
created_count = 0
merged_count = 0
total_count = len(nodes_keyword_df)

# 진행 상황 출력 간격
print_interval = 100

print(f"전체 Keyword 개수: {total_count}")

with driver.session() as session:
    for idx, row in tqdm(nodes_keyword_df.iterrows(), total=total_count):
        name = row['name']
        link = row['link']
        aliases = row['alias']
        categories = row['categories']
        
        # Keyword 처리 (병합 또는 생성)
        result = session.execute_write(
            process_keyword, 
            name=name, 
            link=link, 
            aliases=aliases, 
            categories=categories
        )
        
        # 통계 업데이트
        if result['action'] == 'created':
            created_count += 1
        elif result['action'] == 'merged':
            merged_count += 1
        
        # 진행 상황 출력
        if (idx + 1) % print_interval == 0:
            print(f"진행: {idx + 1}/{total_count} | 생성: {created_count}, 병합: {merged_count}")

print(f"\n=== 처리 완료 ===")
print(f"총 처리: {total_count}개")
print(f"새로 생성: {created_count}개")
print(f"기존 노드에 병합: {merged_count}개")

### 6-4. Paper Node 생성

In [ ]:
def create_paper_node_batch(tx, nodes_batch):
    query = """
    UNWIND $nodes as node
    MERGE (p:Paper {name: node.name})
    SET p.paperId = node.paperId,
        p.categories = node.categories,
        p.year = node.year,
        p.url = node.url,
        p.authors = node.authors,
        p.abstract = node.abstract,
        p.publication = node.publication,
        p.referenceCount = node.referenceCount,
        p.citationCount = node.citationCount,
        p.description = node.description
    """
    tx.run(query, nodes=nodes_batch)

In [ ]:
batch_size = 1000
total_nodes = len(nodes_paper_df)

with driver.session() as session:
    for i in range(0, total_nodes, batch_size):
        batch = nodes_paper_df.iloc[i:i+batch_size]
        nodes_batch = [
            {
                'paperId': row['paperId'],
                'year': row['year'],
                'url': row['url'],
                'name': row['title'],
                'authors': row['authors'],
                'abstract': row['abstract'],
                'publication': row['publication'],
                'referenceCount': row['referenceCount'],
                'citationCount': row['citationCount'],
                'description': row['description']
            }
            for _, row in batch.iterrows()
        ]
        session.execute_write(create_paper_node_batch, nodes_batch)
        print(f"노드 생성 진행: {min(i+batch_size, total_nodes)}/{total_nodes}")


### 6-5. Alias 기반 Edge 처리

In [ ]:
# ABOUT과 IN으로 분리
about_edges_df = edges_keyword_df[edges_keyword_df['name'] == 'ABOUT']
in_edges_df = edges_keyword_df[edges_keyword_df['name'] == 'IN']

print(f"\nABOUT 관계 (Paper -> Keyword): {len(about_edges_df)}개")
print(f"IN 관계 (Keyword -> Paper): {len(in_edges_df)}개")

In [ ]:
def get_all_alias_mapping(tx) -> Dict[str, str]:
    """
    모든 Keyword 노드의 alias 매핑 생성
    alias(소문자) -> 실제 노드 name 매핑
    """
    query = """
    MATCH (k:Keyword)
    RETURN k.name as name, k.alias as alias
    """
    result = tx.run(query)
    
    alias_map = {}
    for record in result:
        name = record['name']
        aliases = record['alias'] if record['alias'] else []
        
        # name 자체도 키로 추가
        if name:
            key = name.lower().strip()
            alias_map[key] = name
        
        # 모든 alias를 키로 추가
        for alias in aliases:
            if alias:
                key = alias.lower().strip()
                alias_map[key] = name
    
    return alias_map

with driver.session() as session:
    alias_to_keyword_map = session.execute_read(get_all_alias_mapping)

print(f"총 {len(alias_to_keyword_map)}개의 alias 매핑 생성")
print("\n샘플 매핑:")
for i, (alias, keyword_name) in enumerate(list(alias_to_keyword_map.items())[:5]):
    print(f"  '{alias}' -> '{keyword_name}'")

In [ ]:
def prepare_edges_with_mapping(df: pd.DataFrame, alias_map: Dict[str, str]) -> Tuple[List[Dict], List[Dict]]:
    """
    Edge 데이터를 처리하여 실제 노드 name으로 변환
    concept 필드를 alias 매핑으로 실제 Keyword name 찾기
    Returns: (valid_edges, failed_edges)
    """
    valid_edges = []
    failed_edges = []

    for idx, row in df.iterrows():
        paper_id = row['paperId']
        concept = row['concept']  # Keyword의 name 또는 alias
        strength = row['strength']
        reason = row['reason'] if pd.notna(row['reason']) else None

        # concept를 alias 매핑으로 실제 Keyword name 찾기
        concept_key = concept.lower().strip()

        if concept_key in alias_map:
            keyword_name = alias_map[concept_key]
            valid_edges.append({
                'paper_id': paper_id,
                'keyword_name': keyword_name,  # 실제 노드의 name 사용
                'strength': float(strength),
                'reason': reason
            })
        else:
            failed_edges.append({
                'paper_id': paper_id,
                'concept': concept,
                'reason': 'No matching Keyword node found in alias map'
            })

    return valid_edges, failed_edges

# ABOUT 관계 처리 (Paper -> Keyword)
print("ABOUT 관계 처리 중...")
valid_about_edges, failed_about_edges = prepare_edges_with_mapping(about_edges_df, alias_to_keyword_map)

print(f"\n=== ABOUT Edge 처리 결과 ===")
print(f"유효한 Edge: {len(valid_about_edges)}")
print(f"실패한 Edge: {len(failed_about_edges)}")

# IN 관계 처리 (Keyword -> Paper)
print("\nIN 관계 처리 중...")
valid_in_edges, failed_in_edges = prepare_edges_with_mapping(in_edges_df, alias_to_keyword_map)

print(f"\n=== IN Edge 처리 결과 ===")
print(f"유효한 Edge: {len(valid_in_edges)}")
print(f"실패한 Edge: {len(failed_in_edges)}")

# 전체 실패 예시
all_failed = failed_about_edges + failed_in_edges
if all_failed:
    print(f"\n실패 예시 (처음 10개):")
    for i, edge in enumerate(all_failed[:10]):
        print(f"{i+1}. Paper {edge['paper_id']} -> '{edge['concept']}': {edge['reason']}")

### 6-6. Keyword-Paper Edge 생성

In [ ]:
def create_about_edges_batch(tx, edges_batch):
    """
    Paper -> Keyword ABOUT 관계를 배치로 생성
    """
    query = """
    UNWIND $edges as edge
    MATCH (p:Paper {paperId: edge.paper_id})
    MATCH (k:Keyword {name: edge.keyword_name})
    MERGE (p)-[r:ABOUT]->(k)
    SET r.strength = edge.strength,
        r.reason = edge.reason
    """
    tx.run(query, edges=edges_batch)

# ABOUT Edge 생성
batch_size = 1000
total_about = len(valid_about_edges)
about_count = 0

print(f"총 {total_about}개의 ABOUT Edge 생성 시작...\n")

with driver.session() as session:
    for i in range(0, total_about, batch_size):
        batch = valid_about_edges[i:i+batch_size]
        session.execute_write(create_about_edges_batch, batch)
        about_count += len(batch)
        print(f"ABOUT Edge 진행: {min(i+batch_size, total_about)}/{total_about}")

print(f"\n총 {about_count}개의 Paper-[ABOUT]->Keyword Edge 생성 완료")

In [ ]:
def create_in_edges_batch(tx, edges_batch):
    """
    Keyword -> Paper IN 관계를 배치로 생성
    """
    query = """
    UNWIND $edges as edge
    MATCH (k:Keyword {name: edge.keyword_name})
    MATCH (p:Paper {paperId: edge.paper_id})
    MERGE (k)-[r:IN]->(p)
    SET r.strength = edge.strength,
        r.reason = edge.reason
    """
    tx.run(query, edges=edges_batch)

# IN Edge 생성
batch_size = 1000
total_in = len(valid_in_edges)
in_count = 0

print(f"총 {total_in}개의 IN Edge 생성 시작...\n")

with driver.session() as session:
    for i in range(0, total_in, batch_size):
        batch = valid_in_edges[i:i+batch_size]
        session.execute_write(create_in_edges_batch, batch)
        in_count += len(batch)
        print(f"IN Edge 진행: {min(i+batch_size, total_in)}/{total_in}")

print(f"\n총 {in_count}개의 Paper-[IN]->Keyword Edge 생성 완료")

### 6-7. Paper-Paper Edge 생성

In [ ]:
def create_ref_relation_batch(tx, edges_batch):
    query = """
    UNWIND $edges as edge
    MATCH (source:Paper {paperId: edge.source_id})
    MATCH (target:Paper {paperId: edge.target_id})
    MERGE (source)-[r:REF_BY]->(target)
    SET r.intents = edge.intents,
        r.isInfluential = edge.isInfluential,
        r.strength = edge.strength,
        r.contexts = edge.contexts
    """
    tx.run(query, edges=edges_batch)

In [ ]:
batch_size = 1000
total_edges = len(edges_paper_df)

with driver.session() as session:
    for i in range(0, total_edges, batch_size):
        batch = edges_paper_df.iloc[i:i+batch_size]
        edges_batch = [
            {
                'source_id': row['ref_paper_id'],
                'target_id': row['seed_paper_id'],
                'intents': row['intents'],
                'isInfluential': row['isInfluential'],
                'contexts': row['contexts'],
                'strength': row['strength']
            }
            for _, row in batch.iterrows()
        ]
        session.execute_write(create_ref_relation_batch, edges_batch)
        print(f"엣지 생성 진행: {min(i+batch_size, total_edges)}/{total_edges}")

print(f"\n총 생성 엣지: {total_edges}")